In [ ]:
# ============================================
# COCO 2017: ANOMALY DETECTION + CLEAN MASK EXTRACTION
# Filters bad images, extracts valid masks only -> Ready for any segmentation model
# ============================================

# importing reuired libraries
from pathlib import Path
import json
import numpy as np
from PIL import Image, ImageFile
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
from tqdm import tqdm

# Fix PIL truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ============================================
# CONFIG (Kaggle-ready)
# ============================================
DATA_ROOT = Path("/kaggle/input/coco-2017-dataset/coco2017")

IMG_ROOT = {
    "train": DATA_ROOT / "train2017",
    "val": DATA_ROOT / "val2017",
}

ANN_FILE = {
    "train": DATA_ROOT / "annotations" / "instances_train2017.json",
    "val": DATA_ROOT / "annotations" / "instances_val2017.json",
}

TARGET_SIZE = (640, 640)
MIN_BOX_AREA = 32.0  # 5.7x5.7 pixels minimum

# ============================================
# STEP 1: ANOMALY DETECTION & VALIDATION
# ============================================

def validate_coco_split(split: str, save_bad_list: bool = True):
    """Find anomalies: missing images, tiny objects, invalid masks, corrupted"""
    print(f"\n🔍 Validating {split} dataset...")
    coco = COCO(str(ANN_FILE[split]))
    img_ids = coco.getImgIds()
    
    anomalies = {
        "missing_images": [],
        "tiny_boxes": [],
        "invalid_masks": [],
        "corrupted_images": [],
        "empty_annotations": []
    }
    
    valid_img_ids = []
    
    for img_id in tqdm(img_ids, desc=f"Checking {split}", leave=False):
        try:
            img_meta = coco.loadImgs([img_id])[0]
            img_path = IMG_ROOT[split] / img_meta["file_name"]
            
            # Check 1: Missing file
            if not img_path.exists():
                anomalies["missing_images"].append(img_id)
                continue
                
            # Check 2: Corrupted image
            with Image.open(img_path) as im:
                img = np.asarray(im.convert("RGB"))
                if img.size == 0 or img.shape[0] == 0:
                    anomalies["corrupted_images"].append(img_id)
                    continue
                    
            # Check 3: No annotations
            ann_ids = coco.getAnnIds(imgIds=[img_id])
            anns = coco.loadAnns(ann_ids)
            if len(anns) == 0:
                anomalies["empty_annotations"].append(img_id)
                continue
                
            # Check 4: All tiny boxes
            valid_anns = [ann for ann in anns if ann["area"] >= MIN_BOX_AREA]
            if len(valid_anns) == 0:
                anomalies["tiny_boxes"].append(img_id)
                continue
                
            # Check 5: All invalid masks
            invalid_mask_count = 0
            for ann in valid_anns:
                if "segmentation" not in ann or not ann["segmentation"]:
                    invalid_mask_count += 1
                    continue
                
                try:
                    h0, w0 = img.shape[:2]
                    if isinstance(ann["segmentation"], list):
                        rles = maskUtils.frPyObjects(ann["segmentation"], h0, w0)
                        rle = maskUtils.merge(rles)
                    else:
                        rle = ann["segmentation"]
                    mask = maskUtils.decode(rle)
                    if mask.sum() == 0:
                        invalid_mask_count += 1
                except:
                    invalid_mask_count += 1
            
            if invalid_mask_count == len(valid_anns):
                anomalies["invalid_masks"].append(img_id)
                continue
            
            valid_img_ids.append(img_id)
            
        except Exception:
            anomalies["corrupted_images"].append(img_id)
            continue
    
    # Save detailed report
    if save_bad_list:
        report = {
            "split": split,
            "total_images": len(img_ids),
            "valid_images": len(valid_img_ids),
            "percent_valid": f"{100*len(valid_img_ids)/len(img_ids):.1f}%",
            "anomalies": {k: len(v) for k, v in anomalies.items()},
            "valid_img_ids": valid_img_ids[:100],  # Sample
            "bad_details": anomalies
        }
        report_path = Path(f"/kaggle/working/anomaly_report_{split}.json")
        report_path.write_text(json.dumps(report, indent=2))
    
    print(f"✅ {split.upper()}: {len(valid_img_ids)}/{len(img_ids)} VALID ({100*len(valid_img_ids)/len(img_ids):.1f}%)")
    return valid_img_ids

# ============================================
# STEP 2: EXTRACT CLEAN MASKS ONLY
# ============================================

def extract_clean_masks(split: str, valid_img_ids: list, out_dir: Path):
    """Create multi-class mask PNGs (pixel value = COCO category ID 0-80)"""
    print(f"\n🎭 Extracting {len(valid_img_ids)} clean masks for {split}...")
    out_dir.mkdir(parents=True, exist_ok=True)
    coco = COCO(str(ANN_FILE[split]))
    
    saved_count = 0
    for img_id in tqdm(valid_img_ids, desc=f"Creating {split} masks", leave=False):
        try:
            img_meta = coco.loadImgs([img_id])[0]
            img_path = IMG_ROOT[split] / img_meta["file_name"]
            
            # Get original dimensions (no full image load)
            with Image.open(img_path) as im:
                h0, w0 = im.size[1], im.size[0]  # PIL: (W,H) → (H,W)
            
            ann_ids = coco.getAnnIds(imgIds=[img_id])
            anns = coco.loadAnns(ann_ids)
            
            # Multi-class mask: 0=background, 1-80=COCO categories
            mask_clean = np.zeros((h0, w0), dtype=np.uint16)
            
            for ann in anns:
                if ann["area"] < MIN_BOX_AREA or not ann.get("segmentation"):
                    continue
                
                cat_id = ann["category_id"]
                
                try:
                    if isinstance(ann["segmentation"], list):
                        rles = maskUtils.frPyObjects(ann["segmentation"], h0, w0)
                        rle = maskUtils.merge(rles)
                    else:
                        rle = ann["segmentation"]
                    
                    binary_mask = maskUtils.decode(rle).astype(np.uint8)
                    
                    # Overwrite with category ID (instance → semantic)
                    mask_clean[binary_mask > 0] = cat_id
                    
                except Exception:
                    continue
            
            # Save only if has valid masks
            if mask_clean.max() > 0:
                out_path = out_dir / f"{img_meta['file_name'].split('.')[0]}_mask.png"
                Image.fromarray(mask_clean).save(out_path, "PNG", compress_level=6)
                saved_count += 1
                
        except Exception as e:
            print(f"⚠️ Failed {img_meta.get('file_name', img_id)}: {e}")
            continue
    
    print(f"✅ Saved {saved_count}/{len(valid_img_ids)} clean mask PNGs")
    return saved_count

# ============================================
# STEP 3: RUN FULL PIPELINE
# ============================================

if __name__ == "__main__":
    print("🚀 COCO CLEAN MASK PIPELINE STARTED")
    
    # 1. Find valid images (filter anomalies)
    print("\n" + "="*50)
    valid_train_ids = validate_coco_split("train")
    valid_val_ids = validate_coco_split("val")
    
    # 2. Extract clean masks
    MASK_OUTPUT = Path("/kaggle/working/masks")
    train_saved = extract_clean_masks("train", valid_train_ids, MASK_OUTPUT / "train")
    val_saved = extract_clean_masks("val", valid_val_ids, MASK_OUTPUT / "val")
    
    # 3. Final summary
    print("\n" + "="*50)
    print("🎉 PIPELINE COMPLETE!")
    print(f"📁 Output: {MASK_OUTPUT}")
    print(f"   Train masks: {train_saved}")
    print(f"   Val masks:   {val_saved}")
    print(f"📊 Reports: anomaly_report_train.json, anomaly_report_val.json")
    print("\n✅ Ready for training segmentation model!")


🚀 COCO CLEAN MASK PIPELINE STARTED


🔍 Validating train dataset...
loading annotations into memory...
Done (t=17.89s)
creating index...
index created!


✅ TRAIN: 117254/118287 VALID (99.1%)

🔍 Validating val dataset...
loading annotations into memory...
Done (t=0.67s)
creating index...
index created!


✅ VAL: 4952/5000 VALID (99.0%)

🎭 Extracting 117254 clean masks for train...
loading annotations into memory...
Done (t=11.51s)
creating index...
index created!


✅ Saved 117254/117254 clean mask PNGs

🎭 Extracting 4952 clean masks for val...
loading annotations into memory...
Done (t=0.42s)
creating index...
index created!


✅ Saved 4952/4952 clean mask PNGs

🎉 PIPELINE COMPLETE!
📁 Output: /kaggle/working/masks
   Train masks: 117254
   Val masks:   4952
📊 Reports: anomaly_report_train.json, anomaly_report_val.json

✅ Ready for training segmentation model!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm

# ============================================
# 1. MODEL ARCHITECTURE (DeepLabV3+)
# ============================================

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.b0 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU())
        self.b1 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=6, dilation=6, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU())
        self.b2 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU())
        self.b3 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=18, dilation=18, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU())
        
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), 
                                  nn.Conv2d(in_channels, out_channels, 1, bias=False), 
                                  nn.BatchNorm2d(out_channels), nn.ReLU())

        self.project = nn.Sequential(nn.Conv2d(5 * out_channels, out_channels, 1, bias=False), 
                                     nn.BatchNorm2d(out_channels), nn.ReLU(), nn.Dropout(0.5))

    def forward(self, x):
        size = x.shape[-2:]
        feat0 = self.b0(x)
        feat1 = self.b1(x)
        feat2 = self.b2(x)
        feat3 = self.b3(x)
        feat_pool = F.interpolate(self.pool(x), size=size, mode='bilinear', align_corners=False)
        return self.project(torch.cat([feat0, feat1, feat2, feat3, feat_pool], dim=1))

class DeepLabV3Plus(nn.Module):
    def __init__(self, n_classes=91):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.backbone_low = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1) 
        self.backbone_high = nn.Sequential(resnet.layer2, resnet.layer3, resnet.layer4) 
        self.aspp = ASPP(2048, 256)
        self.shortcut_conv = nn.Sequential(nn.Conv2d(256, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU())
        self.decoder = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU()
        )
        self.classifier = nn.Conv2d(256, n_classes, 1)

    def forward(self, x):
        size = x.shape[-2:]
        low_feat = self.backbone_low(x)
        high_feat = self.backbone_high(low_feat)
        x = self.aspp(high_feat)
        x = F.interpolate(x, size=low_feat.shape[-2:], mode='bilinear', align_corners=False)
        low_feat = self.shortcut_conv(low_feat)
        x = torch.cat([x, low_feat], dim=1)
        x = self.decoder(x)
        x = self.classifier(x)
        return F.interpolate(x, size=size, mode='bilinear', align_corners=False)

# ============================================
# 2. METRICS & VALIDATION
# ============================================

class SegmentationMetrics:
    def __init__(self, n_classes):
        self.n_classes = n_classes

    def calculate(self, preds, masks):
        preds = torch.argmax(preds, dim=1)
        iou_list, dice_list = [], []
        
        for cls in range(self.n_classes):
            pred_inds = (preds == cls)
            target_inds = (masks == cls)
            intersection = (pred_inds & target_inds).float().sum()
            union = (pred_inds | target_inds).float().sum()
            
            iou = intersection / (union + 1e-6)
            dice = (2 * intersection) / (pred_inds.float().sum() + target_inds.float().sum() + 1e-6)
            
            iou_list.append(iou.item())
            dice_list.append(dice.item())
        return np.mean(iou_list), np.mean(dice_list)

def validate(model, loader, criterion, device, n_classes):
    model.eval()
    val_loss, total_iou, total_dice = 0, 0, 0
    metrics = SegmentationMetrics(n_classes)
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            with torch.amp.autocast('cuda'):
                preds = model(imgs)
                loss = criterion(preds, masks)
            val_loss += loss.item()
            miou, mdice = metrics.calculate(preds, masks)
            total_iou += miou; total_dice += mdice
    return val_loss/len(loader), total_iou/len(loader), total_dice/len(loader)

# ============================================
# 3. DATASET & TRAINING
# ============================================

class COCOMaskDataset(Dataset):
    def __init__(self, mask_dir, img_dir, img_size=256, train=True):
        self.mask_dir, self.img_dir = Path(mask_dir), Path(img_dir)
        self.img_size, self.train = img_size, train
        self.mask_files = sorted(list(self.mask_dir.glob("*_mask.png")))
        self.img_files = [self.img_dir / f"{f.stem.replace('_mask', '')}.jpg" for f in self.mask_files]
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    def __len__(self): return len(self.img_files)
    def __getitem__(self, idx):
        img = Image.open(self.img_files[idx]).convert("RGB").resize((self.img_size, self.img_size))
        mask = Image.open(self.mask_files[idx]).resize((self.img_size, self.img_size), Image.NEAREST)
        if self.train and np.random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT); mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        img_np = (np.array(img).transpose(2, 0, 1) / 255.0 - self.mean) / self.std
        return torch.tensor(img_np, dtype=torch.float32), torch.tensor(np.array(mask), dtype=torch.long)

def run_training():
    MASK_BASE = Path("/kaggle/working/masks")
    IMG_BASE = Path("/kaggle/input/coco-2017-dataset/coco2017")
    N_CLASSES = 91
    
    train_loader = DataLoader(COCOMaskDataset(MASK_BASE/"train", IMG_BASE/"train2017"), batch_size=16, shuffle=True)
    val_loader = DataLoader(COCOMaskDataset(MASK_BASE/"val", IMG_BASE/"val2017", train=False), batch_size=16)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DeepLabV3Plus(n_classes=N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    scaler = torch.amp.GradScaler('cuda')
    best_iou = 0.0

    for epoch in range(10):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for imgs, masks in pbar:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                preds = model(imgs); loss = criterion(preds, masks)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        v_loss, v_iou, v_dice = validate(model, val_loader, criterion, device, N_CLASSES)
        print(f"Val IoU: {v_iou:.4f} | Val Dice: {v_dice:.4f}")
        if v_iou > best_iou:
            best_iou = v_iou
            torch.save(model.state_dict(), "best_deeplabv3plus.pth")
            print(f"⭐ Best Model Saved!")

if __name__ == "__main__":
    run_training()

Epoch 1:   3%|▎         | 222/7329 [01:21<44:12,  2.68it/s, loss=1.3794]